# Module 5: An LLM as the judge

A general-purpose LLM, prompted with a policy you write, decides whether each prompt may pass:
NeMo Guardrails' `self check input` rail. Each section follows a terminal step in the guide; run a
section only after its terminal step. The judge runs remotely and takes several seconds per
prompt: a full 50-prompt run takes about five minutes.

## 1. Setup

In [ ]:
import json, ssl, urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import labeval as lab  # the evaluation code, next to this notebook (also used by the lab's tests)

NS = lab.namespace()
MONITOR = lab.monitor_url()
rows = lab.load_rows()
ten = lab.load_rows(stride=5)  # every fifth prompt: four unsafe, six safe
print(f"namespace: {NS}\nmonitor:   {MONITOR}")
print("guardrails configs:", lab.require_configs("regex-only", "content-safety", "self-check"))

In [ ]:
def tick(line):  # one character per prompt: . correct, x wrong, ! no verdict
    outcome, verdict = line.split()[1], line.split()[2]
    print("!" if verdict == "error" else "." if outcome in ("TP", "TN") else "x", end="", flush=True)

def metrics_table(*results):
    return pd.DataFrame([{"run": r["run"], "config": r["config"], "errors as": r["errors_as"],
                          **r["counts"], "errors": r["errors"],
                          **{k: round(v, 2) for k, v in r["metrics"].items()}} for r in results]).set_index("run")

def confusion(ax, res):
    c = res["counts"]
    grid = [[c["TN"], c["FP"]], [c["FN"], c["TP"]]]
    ax.imshow(grid, cmap="Blues")
    for i, row in enumerate(grid):
        for j, v in enumerate(row):
            ax.text(j, i, f"{['TN', 'FP', 'FN', 'TP'][2 * i + j]}\n{v}", ha="center", va="center", fontsize=13,
                    color="white" if v > max(map(max, grid)) / 2 else "black")
    ax.set_xticks([0, 1], ["allowed", "blocked"]); ax.set_yticks([0, 1], ["safe", "unsafe"])
    ax.set_xlabel("guardrails verdict"); ax.set_ylabel("label")
    m = res["metrics"]
    ax.set_title(f"{res['config']}\nP={m['precision']:.2f} R={m['recall']:.2f} FPR={m['fpr']:.2f}")

def wrong(res):
    return pd.DataFrame([r for r in res["results"] if r["outcome"] in ("FP", "FN")],
                        columns=["outcome", "id", "prompt", "rail", "categories"])

## 2. After Exercise 1: a first measurement

Ten prompts through the judge with the policy prompt as written. Then every safe prompt it blocked,
asked once more, with the judge's own words next to the verdict NeMo derived from them.

In [ ]:
first = lab.evaluate("self-check", ten, progress=tick)
print("\n" + lab.summary(first))
assert first["errors"] == 0, "prompts without a verdict: read the WARNING above before comparing anything"

In [ ]:
# the safe prompts it blocked, asked again: NeMo's verdict, then the judge's own answer (whitespace folded)
for r in first["results"]:
    if r["outcome"] == "FP":
        e = lab.explain("self-check", r["prompt"])
        said = " ".join(" ".join(call["answer"] or "" for call in e["llm_calls"]).split())
        print(f"{e['verdict']:8s} {e['prompt']}\n         judge: {said[:200]}\n")

## 3. After Exercise 3: the judge answers in one word

All 50 prompts, then the three judges side by side: patterns, the fine-tuned safety model, and the
prompted LLM. Latency compares the two models on the same ten prompts.

In [ ]:
judge = lab.evaluate("self-check", rows, progress=tick)
print("\n" + lab.summary(judge))
assert judge["errors"] == 0, "prompts without a verdict: read the WARNING above before comparing anything"

In [ ]:
runs = lab.monitor_runs()
baselines = []
for config in ("regex-only", "content-safety"):  # your Module 3 runs (error-free only); measured again if there are none
    b = lab.latest_run(runs, config, len(rows))
    if b is None:
        b = lab.evaluate(config, rows, progress=tick)
        print("\n" + lab.summary(b))
    assert b["errors"] == 0, f"the {config} baseline had prompts without a verdict: read the WARNING above"
    baselines.append(b)
metrics_table(*baselines, judge)

In [ ]:
cs_ten = lab.evaluate("content-safety", ten, progress=tick)
assert cs_ten["errors"] == 0, "content-safety gave no verdict: these would be error timings, not model latency"
judge_ten = [r for r in judge["results"] if r["id"] in {t["id"] for t in ten}]
latency = pd.DataFrame({"content-safety (Nemotron, CPU)": [r["seconds"] for r in cs_ten["results"]],
                        "self-check (LLM judge, MaaS)": [r["seconds"] for r in judge_ten]})
print("\nseconds per prompt on the same ten prompts")
latency.describe().loc[["min", "50%", "max"]].round(1)

In [ ]:
wrong(judge)

## 4. After Exercise 4: the policy edited

The same 50 prompts through `self-check-edited`, the policy with the two new lines. Which verdicts
moved, and did they move because of the edit or by chance? The original policy is still live as
`self-check`, so each moved prompt can be asked again under both.

In [ ]:
lab.require_configs("self-check-edited")
edited = lab.evaluate("self-check-edited", rows, progress=tick)
print("\n" + lab.summary(edited))
assert edited["errors"] == 0, "prompts without a verdict: read the WARNING above before comparing anything"
metrics_table(judge, edited)

In [ ]:
before = {r["id"]: r["outcome"] for r in judge["results"]}
pd.DataFrame([{"id": r["id"], "label": r["label"], "before": before[r["id"]], "after": r["outcome"], "prompt": r["prompt"]}
              for r in edited["results"] if before[r["id"]] != r["outcome"]])

In [ ]:
# moved by the edit, or by chance? Ask every moved prompt three more times under each policy
moved = [r for r in edited["results"] if before[r["id"]] != r["outcome"]]
again = {}
for config in ("self-check", "self-check-edited"):
    again[config] = lab.evaluate(config, [r for r in moved for _ in range(3)], progress=tick)
    assert again[config]["errors"] == 0, "prompts without a verdict: read the WARNING above before comparing anything"
blocked = lambda config, i: sum(a["verdict"] == "blocked" for a in again[config]["results"] if a["id"] == i)
pd.DataFrame([{"id": r["id"], "label": r["label"], "before": before[r["id"]], "after": r["outcome"],
               "original: blocked (of 3)": blocked("self-check", r["id"]),
               "edited: blocked (of 3)": blocked("self-check-edited", r["id"]), "prompt": r["prompt"]} for r in moved])